# Data Cleaning: KwaZulu-Natal Afrobarometer 2024

This notebook cleans and validates the KwaZulu-Natal Afrobarometer 2024 voter attitudes dataset.

The raw dataset is kept unchanged in `data/raw/`. The cleaned dataset is saved separately in `data/interim/`.

The cleaning process checks for:
- Missing values
- Duplicate records
- Inconsistent response values
- Leading or trailing spaces
- Empty or junk rows
- Column structure
- Data types
- Location information
- Preservation of records after cleaning


In [1]:
import pandas as pd
from pathlib import Path

In [6]:

# Notebook location:
# SPU-TEAM-DIRISA/notebooks/Data Collection/

project_root = Path("../..")

raw_file = (
    project_root
    / "data"
    / "raw"
    / "KZN_Voting_Political_Attitudes_Afrobarometer-2024"
    / "kzn_afrobarometer_round9_readable_voter_data.csv"
)

interim_folder = (
    project_root
    / "data"
    / "interim"
    / "KZN_Voting_Political_Attitudes_Afrobarometer-2024"
)

interim_folder.mkdir(parents=True, exist_ok=True)

cleaned_file = (
    interim_folder
    / "kzn_afrobarometer_round9_readable_voter_data.csv"
)

print("Raw file:", raw_file)
print("Raw file exists:", raw_file.exists())
print("Interim folder:", interim_folder)

Raw file: ..\..\data\raw\KZN_Voting_Political_Attitudes_Afrobarometer-2024\kzn_afrobarometer_round9_readable_voter_data.csv
Raw file exists: True
Interim folder: ..\..\data\interim\KZN_Voting_Political_Attitudes_Afrobarometer-2024


In [7]:
data = pd.read_csv(
    raw_file,
    sep=";",
    encoding="utf-8"
)

print("Dataset loaded successfully.")
print("Number of respondents:", len(data))
print("Number of variables:", len(data.columns))

Dataset loaded successfully.
Number of respondents: 248
Number of variables: 14


In [8]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 248 entries, 0 to 247
Data columns (total 14 columns):
 #   Column                                                                 Non-Null Count  Dtype
---  ------                                                                 --------------  -----
 0   Q13. Voting in the most recent national election                       248 non-null    str  
 1   Q9c. Freedom to choose who to vote for                                 248 non-null    str  
 2   Q12a. Elections ensure MPs reflect views of voters                     248 non-null    str  
 3   Q12b. Elections ensure voters remove unrepresentative leaders          248 non-null    str  
 4   Q14a. Freeness and fairness of the last national election              248 non-null    str  
 5   Q14b. How likely powerful find out your vote                           248 non-null    str  
 6   Q14c. Last national election: fear political intimidation or violence  248 non-null    str  
 7   Q19b. Elected offic

In [9]:
print("Number of columns:", len(data.columns))

for number, column in enumerate(data.columns, start=1):
    print(f"{number}. {column}")

Number of columns: 14
1. Q13. Voting in the most recent national election
2. Q9c. Freedom to choose who to vote for
3. Q12a. Elections ensure MPs reflect views of voters
4. Q12b. Elections ensure voters remove unrepresentative leaders
5. Q14a. Freeness and fairness of the last national election
6. Q14b. How likely powerful find out your vote
7. Q14c. Last national election: fear political intimidation or violence
8. Q19b. Elected officials follow voter demands vs follow own ideas
9. Q24. Choose leaders through elections vs other methods
10. Q23b-saf. Willing to give up regular elections
11. Urban or Rural Primary Sampling Unit
12. Q3. Overall direction of the country
13. Q4a. Country’s present economic condition
14. Q4b. Your present living conditions


In [10]:
missing_values = data.isnull().sum()

print("Missing values by column:")
print(missing_values)

print("\nTotal missing values:", missing_values.sum())

Missing values by column:
Q13. Voting in the most recent national election                         0
Q9c. Freedom to choose who to vote for                                   0
Q12a. Elections ensure MPs reflect views of voters                       0
Q12b. Elections ensure voters remove unrepresentative leaders            0
Q14a. Freeness and fairness of the last national election                0
Q14b. How likely powerful find out your vote                             0
Q14c. Last national election: fear political intimidation or violence    0
Q19b. Elected officials follow voter demands vs follow own ideas         0
Q24. Choose leaders through elections vs other methods                   0
Q23b-saf. Willing to give up regular elections                           0
Urban or Rural Primary Sampling Unit                                     0
Q3. Overall direction of the country                                     0
Q4a. Country’s present economic condition                                0

In [11]:
blank_values = (
    data.astype(str)
    .apply(lambda column: column.str.strip().eq(""))
    .sum()
)

print("Blank values by column:")
print(blank_values)

print("\nTotal blank values:", blank_values.sum())

Blank values by column:
Q13. Voting in the most recent national election                         0
Q9c. Freedom to choose who to vote for                                   0
Q12a. Elections ensure MPs reflect views of voters                       0
Q12b. Elections ensure voters remove unrepresentative leaders            0
Q14a. Freeness and fairness of the last national election                0
Q14b. How likely powerful find out your vote                             0
Q14c. Last national election: fear political intimidation or violence    0
Q19b. Elected officials follow voter demands vs follow own ideas         0
Q24. Choose leaders through elections vs other methods                   0
Q23b-saf. Willing to give up regular elections                           0
Urban or Rural Primary Sampling Unit                                     0
Q3. Overall direction of the country                                     0
Q4a. Country’s present economic condition                                0
Q

In [12]:
duplicate_count = data.duplicated().sum()

print("Duplicate complete respondent records:", duplicate_count)

Duplicate complete respondent records: 0


In [13]:
extra_spaces = {}

for column in data.columns:
    extra_spaces[column] = (
        data[column]
        .astype(str)
        .ne(data[column].astype(str).str.strip())
        .sum()
    )

extra_spaces = pd.Series(extra_spaces)

print("Values with leading or trailing spaces:")
print(extra_spaces)

print("\nTotal values with extra spaces:", extra_spaces.sum())

Values with leading or trailing spaces:
Q13. Voting in the most recent national election                         0
Q9c. Freedom to choose who to vote for                                   0
Q12a. Elections ensure MPs reflect views of voters                       0
Q12b. Elections ensure voters remove unrepresentative leaders            0
Q14a. Freeness and fairness of the last national election                0
Q14b. How likely powerful find out your vote                             0
Q14c. Last national election: fear political intimidation or violence    0
Q19b. Elected officials follow voter demands vs follow own ideas         0
Q24. Choose leaders through elections vs other methods                   0
Q23b-saf. Willing to give up regular elections                           0
Urban or Rural Primary Sampling Unit                                     0
Q3. Overall direction of the country                                     0
Q4a. Country’s present economic condition                   

In [14]:
for column in data.columns:
    print("\n" + "=" * 80)
    print(column)
    print("=" * 80)
    
    for value in sorted(data[column].dropna().unique().astype(str)):
        print(value)


Q13. Voting in the most recent national election
I did not vote
I voted in the election
I was too young to vote
Refused

Q9c. Freedom to choose who to vote for
Completely free
Don’t know
Not at all free
Not very free
Somewhat free

Q12a. Elections ensure MPs reflect views of voters
Don’t know
Fairly well
Not at all well
Not very well
Very well

Q12b. Elections ensure voters remove unrepresentative leaders
Don’t know
Fairly well
Not at all well
Not very well
Very well

Q14a. Freeness and fairness of the last national election
Completely free and fair
Do not understand question
Don’t know
Free and fair, but with minor problems
Free and fair, with major problems
Not free and fair

Q14b. How likely powerful find out your vote
Don’t know
Not at all likely
Not very likely
Somewhat likely
Very likely

Q14c. Last national election: fear political intimidation or violence
A little bit
A lot
Don’t know
Not at all
Refused
Somewhat

Q19b. Elected officials follow voter demands vs follow own ideas

In [15]:
unique_values = data.nunique()

print("Number of unique responses per variable:")
print(unique_values)

Number of unique responses per variable:
Q13. Voting in the most recent national election                         4
Q9c. Freedom to choose who to vote for                                   5
Q12a. Elections ensure MPs reflect views of voters                       5
Q12b. Elections ensure voters remove unrepresentative leaders            5
Q14a. Freeness and fairness of the last national election                6
Q14b. How likely powerful find out your vote                             5
Q14c. Last national election: fear political intimidation or violence    6
Q19b. Elected officials follow voter demands vs follow own ideas         6
Q24. Choose leaders through elections vs other methods                   6
Q23b-saf. Willing to give up regular elections                           6
Urban or Rural Primary Sampling Unit                                     2
Q3. Overall direction of the country                                     3
Q4a. Country’s present economic condition                  

In [16]:
print("First 3 respondents:")
display(data.head(3))

print("\nLast 3 respondents:")
display(data.tail(3))

First 3 respondents:


,Q13. Voting in the most recent national election,Q9c. Freedom to choose who to vote for,Q12a. Elections ensure MPs reflect views of voters,Q12b. Elections ensure voters remove unrepresentative leaders,Q14a. Freeness and fairness of the last national election,Q14b. How likely powerful find out your vote,Q14c. Last national election: fear political intimidation or violence,Q19b. Elected officials follow voter demands vs follow own ideas,Q24. Choose leaders through elections vs other methods,Q23b-saf. Willing to give up regular elections,Urban or Rural Primary Sampling Unit,Q3. Overall direction of the country,Q4a. Country’s present economic condition,Q4b. Your present living conditions
0,I voted in the election,Not very free,Not at all well,Not at all well,Not free and fair,Don’t know,Somewhat,Agree very strongly with 2,Agree very strongly with 2,Very willing,Urban,Going in the wrong direction,Very bad,Very bad
1,I did not vote,Completely free,Not at all well,Not at all well,Not free and fair,Very likely,Not at all,Agree with neither,Agree very strongly with 1,Very willing,Urban,Going in the wrong direction,Very bad,Very bad
2,I did not vote,Completely free,Fairly well,Fairly well,Don’t know,Don’t know,Not at all,Agree very strongly with 2,Agree very strongly with 1,Very willing,Urban,Going in the wrong direction,Very bad,Fairly good



Last 3 respondents:


,Q13. Voting in the most recent national election,Q9c. Freedom to choose who to vote for,Q12a. Elections ensure MPs reflect views of voters,Q12b. Elections ensure voters remove unrepresentative leaders,Q14a. Freeness and fairness of the last national election,Q14b. How likely powerful find out your vote,Q14c. Last national election: fear political intimidation or violence,Q19b. Elected officials follow voter demands vs follow own ideas,Q24. Choose leaders through elections vs other methods,Q23b-saf. Willing to give up regular elections,Urban or Rural Primary Sampling Unit,Q3. Overall direction of the country,Q4a. Country’s present economic condition,Q4b. Your present living conditions
245,I voted in the election,Completely free,Fairly well,Fairly well,"Free and fair, but with minor problems",Not at all likely,Not at all,Agree with 2,Agree with 2,Very willing,Urban,Going in the right direction,Fairly bad,Very good
246,I did not vote,Completely free,Not very well,Not very well,Don’t know,Very likely,Not at all,Agree with 1,Agree with 2,Very willing,Urban,Going in the wrong direction,Very bad,Fairly good
247,I did not vote,Not very free,Not very well,Not very well,Not free and fair,Somewhat likely,Somewhat,Agree very strongly with 1,Agree very strongly with 2,Willing,Urban,Going in the wrong direction,Fairly good,Fairly bad


In [17]:
empty_rows = data.isnull().all(axis=1).sum()

print("Completely empty rows:", empty_rows)

Completely empty rows: 0


In [18]:
print("Data types:")
print(data.dtypes)

Data types:
Q13. Voting in the most recent national election                         str
Q9c. Freedom to choose who to vote for                                   str
Q12a. Elections ensure MPs reflect views of voters                       str
Q12b. Elections ensure voters remove unrepresentative leaders            str
Q14a. Freeness and fairness of the last national election                str
Q14b. How likely powerful find out your vote                             str
Q14c. Last national election: fear political intimidation or violence    str
Q19b. Elected officials follow voter demands vs follow own ideas         str
Q24. Choose leaders through elections vs other methods                   str
Q23b-saf. Willing to give up regular elections                           str
Urban or Rural Primary Sampling Unit                                     str
Q3. Overall direction of the country                                     str
Q4a. Country’s present economic condition                       

In [19]:
location_columns = [
    column for column in data.columns
    if any(
        word in column.lower()
        for word in ["province", "municipality", "ward", "district"]
    )
]

print("Location-related columns found:")

if location_columns:
    for column in location_columns:
        print(column)
else:
    print("No province, municipality, ward, or district column found.")

Location-related columns found:
No province, municipality, ward, or district column found.


In [20]:
cleaned_data = data.copy()

for column in cleaned_data.columns:
    cleaned_data[column] = (
        cleaned_data[column]
        .astype(str)
        .str.strip()
    )

print("Cleaned dataset created.")

Cleaned dataset created.


In [21]:
cleaned_data.to_csv(
    cleaned_file,
    sep=";",
    index=False,
    encoding="utf-8"
)

print("Cleaned dataset saved successfully.")
print("Saved to:", cleaned_file)

Cleaned dataset saved successfully.
Saved to: ..\..\data\interim\KZN_Voting_Political_Attitudes_Afrobarometer-2024\kzn_afrobarometer_round9_readable_voter_data.csv


In [22]:
validated_data = pd.read_csv(
    cleaned_file,
    sep=";",
    encoding="utf-8"
)

print("Cleaned dataset reloaded successfully.")
print("Rows:", len(validated_data))
print("Columns:", len(validated_data.columns))

Cleaned dataset reloaded successfully.
Rows: 248
Columns: 14


In [23]:
print(
    "Total missing values after cleaning:",
    validated_data.isnull().sum().sum()
)

Total missing values after cleaning: 0


In [24]:
print(
    "Duplicate records after cleaning:",
    validated_data.duplicated().sum()
)

Duplicate records after cleaning: 0


In [25]:
print("Original rows:", len(data))
print("Cleaned rows:", len(validated_data))
print("Row counts match:", len(data) == len(validated_data))

Original rows: 248
Cleaned rows: 248
Row counts match: True


In [26]:
print(
    "Column names and order preserved:",
    list(data.columns) == list(validated_data.columns)
)

Column names and order preserved: True


In [27]:
values_changed = not data.equals(validated_data)

print("Were any survey values changed during cleaning?", values_changed)

Were any survey values changed during cleaning? False


In [28]:
print("=" * 60)
print("FINAL CLEANING VALIDATION")
print("=" * 60)

print("Original respondents:", len(data))
print("Cleaned respondents:", len(validated_data))
print("Original variables:", len(data.columns))
print("Cleaned variables:", len(validated_data.columns))
print("Missing values:", validated_data.isnull().sum().sum())
print("Duplicate records:", validated_data.duplicated().sum())
print("Row counts preserved:", len(data) == len(validated_data))
print("Columns preserved:", list(data.columns) == list(validated_data.columns))
print("Survey values changed:", values_changed)
print("Cleaned file exists:", cleaned_file.exists())

FINAL CLEANING VALIDATION
Original respondents: 248
Cleaned respondents: 248
Original variables: 14
Cleaned variables: 14
Missing values: 0
Duplicate records: 0
Row counts preserved: True
Columns preserved: True
Survey values changed: False
Cleaned file exists: True
